# Multi-agentes de IA para análises de revisões de papers


- Testes com elementos do Langchain: prompt template, tools, retrievers, chains e runnable parallel.

- A partir do input do usuário, o mini sistema multi-agentes busca esclarecer a questão sobre o tópico de entrada: um deles responde as questões de forma generalista; outro busca responder usando o contexto disponibilizado (RAG); e outro realiza uma busca por vídeos no Youtube sobre o tema de interesse para complementar a resposta final.


In [1]:
import re
from rich import print
from typing import List, Tuple
from dotenv import load_dotenv
load_dotenv()

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_core.documents import Document



## Explora componentes do Langchain

### Prompt template

In [2]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    template='Who was the greatest scorer in the history of the {interest_sport}?'
)

prompt_template.format(interest_sport='soccer')

'Who was the greatest scorer in the history of the soccer?'

### LLM chat model

In [3]:
from langchain_groq import ChatGroq
import os

llm_model = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0,
    max_tokens=300,
    timeout=None,
    max_retries=2,
    verbose=False
)

In [4]:
system_prompt = 'You are a sports expert assistant. Answer in bullet points and be concise.'
user_prompt = prompt_template.format(interest_sport='soccer')
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_prompt)
]

response = llm_model.invoke(messages)
print(response.content)

* Josef Bican (Austria, Czechoslovakia) - 805 goals in 530 games
* Ferenc Puskás (Hungary, Spain) - 746 goals in 629 games
* Gerd Müller (Germany) - 735 goals in 706 games
* Pelé (Brazil) - 727 goals in 812 games
* Cristiano Ronaldo (Portugal) - 819 goals in 1156 games (active player)
* Lionel Messi (Argentina) - 822 goals in 1024 games (active player) 

Note: The exact number of goals may vary depending on the source.

### Retriever

*retrievers, WikipediaRetriever*

In [5]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever()

docs = retriever.invoke(input='Pelé')
print(docs[0].metadata)

{
    'title': 'Pelé',
    'summary': 'Edson Arantes do Nascimento (Brazilian Portuguese: [ˈɛd(ʒi)sõ(w) aˈɾɐ̃tʃiz du nasiˈmẽtu]; 23 October
1940 – 29 December 2022), better known by his nickname Pelé (Brazilian Portuguese: [peˈlɛ]), was a Brazilian 
professional footballer who played as a forward. Widely regarded as one of the greatest players in history, he was 
among the most successful and popular sports figures of the 20th century. His 1,279 goals in 1,363 games, which 
includes friendlies, is recognised as a Guinness World Record. In 1999, he was named Athlete of the Century by the 
International Olympic Committee and was included in the Time list of the 100 most important people of the 20th 
century. In 2000, Pelé was voted World Player of the Century by the International Federation of Football History & 
Statistics (IFFHS) and was one of the two joint winners of the FIFA Player of the Century, alongside Diego 
Maradona.\nPelé began playing for Santos at age 15 and for the Brazil national team at 16. During his international
career, he won three FIFA World Cup titles: 1958, 1962, and 1970, becoming the only player to do so and the 
youngest to win a World Cup, at just 17 years old. He was nicknamed O Rei (The King) following the 1958 tournament.
With 77 goals in 92 games for Brazil, Pelé held the record as the national team\'s top goalscorer for over fifty 
years. At club level, he is Santos\'s all-time top goalscorer with 643 goals in 659 games. In a golden era for 
Santos, he led the club to the 1962 and 1963 Copa Libertadores, and to the 1962 and 1963 Intercontinental Cup. 
Credited with connecting the phrase "The Beautiful Game" with football, Pelé\'s "electrifying play and penchant for
spectacular goals" made him a global star, and his teams toured internationally to take full advantage of his 
popularity. During his playing days, Pelé was for a period the best-paid athlete in the world. After retiring in 
1977, Pelé was a worldwide ambassador for football and made many acting and commercial ventures. In 2010, he was 
named the honorary president of the New York Cosmos.\nPelé averaged almost a goal per game throughout his career 
and could strike the ball with either foot, as well as being able to anticipate his opponents\' movements. While 
predominantly a striker, he could also be a playmaker, providing assists with his vision and passing ability. He 
would often use his dribbling skills to go past opponents. In Brazil, he was hailed as a national hero for his 
accomplishments in football and for his outspoken support of policies that improve the social conditions of the 
poor. His emergence at the 1958 World Cup, where he became a black global sporting star, was a source of 
inspiration. Throughout his career and in his retirement, Pelé received numerous individual and team awards for his
performance on the field, his record-breaking achievements, and his legacy in the sport. \n\n',
    'source': 'https://en.wikipedia.org/wiki/Pel%C3%A9'
}

### Tools

In [6]:
from youtubesearchpython import VideosSearch
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent


@dataclass
class UserContext:
    search_terms: str

@tool
def get_video_info(runtime: ToolRuntime[UserContext]) -> str:
    '''Consulta Youtube.'''
    search_terms = runtime.context.search_terms
    tool = VideosSearch(search_terms, limit=1)
    res_search = tool.result()
    return res_search


agent = create_agent(
    llm_model,
    tools=[get_video_info],
    context_schema=UserContext,
    system_prompt='You are an assistant that can search videos in Youtube.'
)

/home/msc/miniconda3/envs/rag_env/lib/python3.11/site-packages/youtubesearchpython/handlers/requesthandler.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [7]:
search_terms = 'Goals of Pelé'
result = agent.invoke(
    {
        'messages': [{
            'role': 'user', 
            'content': 'Extract the link, title and description of the video for the following search terms'
        }]
    },
    context=UserContext(search_terms=search_terms)
)

result

{'messages': [HumanMessage(content='Extract the link, title and description of the video for the following search terms', additional_kwargs={}, response_metadata={}, id='7e54cf75-be85-4914-a6d7-d260d7abd9c5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pna9fq1qa', 'function': {'arguments': '{}', 'name': 'get_video_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 223, 'total_tokens': 232, 'completion_time': 0.034818922, 'completion_tokens_details': None, 'prompt_time': 0.011923841, 'prompt_tokens_details': None, 'queue_time': 0.152947141, 'total_time': 0.046742763}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cd7c0-0365-76f0-aba2-11f33856286c-0', tool_calls=[{'name': 'get_video_info', 'args': {}, 'id': 'pna9fq1qa', 'type': 'tool_call'}], invalid_tool_cal

#### Parse the response

In [8]:
import json

res_search = json.loads(result['messages'][2].content)['result'][0]

print(f' . {res_search["title"]} #')
print(f' . Channel: {res_search["channel"]["name"]}')
print(f' . Duration: {res_search["duration"]}s')
print(f' . Views: {res_search["viewCount"]["short"]}')
print(f' . URL: {res_search["link"]}')

. Pele -Top 10 Impossible Goals Ever #

. Channel: Sports 360

. Duration: 3:37s

. Views: 21M views

. URL: https://www.youtube.com/watch?v=WXg8P0u9W9I

### Chains

In [9]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt_template | llm_model | StrOutputParser()

res_chain = chain.invoke({'interest_sport':'chess'})
print(res_chain)

While chess is not typically measured by scoring in the same way as sports like basketball or football, I assume 
you're asking about the player with the highest overall performance or winning percentage.

According to various sources, including chess databases and historians, the greatest scorer in the history of chess
is often considered to be Garry Kasparov. Kasparov, a Russian chess grandmaster, is widely regarded as one of the 
greatest chess players of all time. He was the world chess champion from 1985 to 1993 and again from 1993 to 2000.

Kasparov's impressive career statistics include:

* A peak Elo rating of 2851, which was the highest in the world at the time
* A career winning percentage of 72.4% in tournament games
* 15 consecutive years as the world's top-ranked player (1986-2000)
* 6 World Chess Championship titles
* Over 200 tournament victories, including 15 Super Tournament wins

Other notable chess players who are often considered among the greatest scorers in history include:

* Emanuel Lasker (1868-1941): A German mathematician and philosopher who was world chess champion from 1894 to 1921
* José Capablanca (1888-1942): A Cuban chess player who was world chess champion from 1921 to 1927
* Bobby Fischer (1943-2008): An American chess player who was world chess

## Chains com múltiplos estágios

#### Chain 1

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


prompt1 = PromptTemplate.from_template(
    template='''Your are a research paper review specialist. Try to give a concise answer about the user question.

    Question: {question_about_reviews}

    Answer:
    '''
)

llm_model1 = ChatGroq(
    model='qwen/qwen3-32b',
    temperature=0,
    max_tokens=100,
    timeout=None,
    max_retries=2,
    verbose=False
)

question_runnable1 = ({
    'question_about_reviews':  RunnablePassthrough(),
})
chain1 = question_runnable1 | prompt1 | llm_model1 | StrOutputParser()

question_about_reviews = 'Can you explain about good practices for the implementation of management systems?'
res_chain1 = chain1.invoke({
    'question_about_reviews': question_about_reviews,
})
print(res_chain1)

<think>
Okay, the user is asking about good practices for implementing management systems. Let me start by recalling what 
management systems typically involve. They're structured frameworks that help organizations manage processes, 
resources, and people to achieve objectives. So, the answer should cover key steps and best practices.

First, leadership commitment is crucial. Without top management support, the system might not get the necessary 
resources or attention. Then, aligning the system with the organization's goals ensures it's relevant and adds 
value.

#### Chain 2

In [11]:
from typing import List
import pickle
import numpy as np

### Carrega vector store e dependências

In [12]:
with open('models/pca_model.pkl', 'rb') as f:
    pca = pickle.load(f)

In [13]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5',
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
def normalize_text(text: str) -> List[float]:
    if text is None:
        return None
    
    t_norm = str(text).strip()
    t_norm = re.sub(r'\s+', ' ', t_norm) 
    t_norm = t_norm.replace('\n', ' ').replace('\r', ' ')
    
    return t_norm


def get_embeddings(text: List[str]) -> List[float]:
    if not text:
        return []
    
    text_norm = normalize_text(text)
    embed_text = embed_model.embed_query(text_norm)

    return embed_text

In [15]:
vs_faiss = FAISS.load_local(
    folder_path='datasets/vectorstore.faiss',
    embeddings=embed_model,
    allow_dangerous_deserialization=True
)


In [16]:
print(f'# Amostras no vector store FAISS: {len(vs_faiss.index_to_docstore_id)}')

# Amostras no vector store FAISS: 6474

### Testa busca semântica

In [17]:
sentence_search = 'prácticas concretas para la implementación de sistemas de gestión'

In [18]:
def run_similarity_search(sentence_search: str, top_k: int = 3) -> List[Tuple[Document, float]]:
    sentence_search_emb_low_dim = np.array([get_embeddings(sentence_search)])
    emb_vector_sent_search = pca.transform(sentence_search_emb_low_dim)
    res = vs_faiss.similarity_search_with_score_by_vector(emb_vector_sent_search[0], k=top_k)
    
    return res

In [19]:
res_search = run_similarity_search(sentence_search=sentence_search)
print(res_search)

[
    (
        Document(
            id='daeb1a0c-612c-4ce8-9d2f-f199752737b1',
            metadata={},
            page_content='ción de conceptos presentes en las normas a un proceso concreto o un e'
        ),
        np.float32(0.21730843)
    ),
    (
        Document(
            id='d502913e-d4ff-4643-aaf9-2dc85999a3f1',
            metadata={},
            page_content='El trabajo presenta una aproximación para la programación de proyectos'
        ),
        np.float32(0.21863703)
    ),
    (
        Document(
            id='7a22e79d-a0f3-4783-952e-806fe7c930b2',
            metadata={},
            page_content='a las organizaciones a construir un modelo para mejorar sus procesos -'
        ),
        np.float32(0.22205812)
    )
]

### Cria chain para busca semântica

In [20]:
import json
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


def str_to_dict(text: str) -> str:
    text = text.strip().strip('"')
    return text

prompt2 = PromptTemplate.from_template(
    template='''
        Use the following context to answer the question: {context}.

        Question: {question_about_topic}.

        Provide a concise answer.

        In addition to the answer, return the context in the following format:
        - References: {context}
    '''
)

llm_model2 = ChatGroq(
    model='qwen/qwen3-32b',
    temperature=0,
    max_tokens=500,
    timeout=None,
    max_retries=2,
    verbose=False
)

question_runnable2 = ({
    'question_about_topic':  RunnablePassthrough(),
    'context': RunnablePassthrough()
})

chain2 = question_runnable2 | prompt2 | llm_model2 | StrOutputParser() | RunnableLambda(str_to_dict)


question_about_topic = 'What paper review information do we have about the following topic?'
selected_topic = 'prácticas concretas para la implementación de sistemas de gestión'

docs_retrieved = run_similarity_search(sentence_search=selected_topic)
context = '\n'.join([doc[0].page_content for doc in docs_retrieved])

system_prompt = 'You are a helpful assistant who answers questions about paper reviews.'

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(
        content=prompt2.format(
            context=context, 
            question_about_topic=question_about_topic,
        )
    )
]

res_chain2 = chain2.invoke(messages)
print(res_chain2)

<think>
Okay, let's see. The user is asking for paper review information based on the provided context. The context is in 
Spanish, so I need to parse that first.

The context mentions "aproximación para la programación de proyectos" which translates to an approach for project 
programming. It also talks about helping organizations build a model to improve their processes. The key points 
here are project programming methods and process improvement models.

The user wants a concise answer about the paper review info. The original context provided by the user includes 
some fragmented sentences, like "ción de conceptos presentes en las normas a un proceso concreto o un e" which 
seems like a typo or incomplete sentence. But the main part is the paper presenting an approach for project 
programming and aiding organizations in creating models for process improvement.

So, the answer should state that the paper introduces an approach for project programming and a model for process 
improvement. The references should list the provided context snippets. I need to make sure the answer is concise 
and only includes the information given, without adding anything extra. Also, check if the references are formatted
correctly as per the user's instructions.
</think>

The paper review information indicates that the work presents an approach for project programming and proposes a 
model to help organizations improve their processes. The context emphasizes applying concepts from standards to 
concrete processes.  

- References: ción de conceptos presentes en las normas a un proceso concreto o un e  
El trabajo presenta una aproximación para la programación de proyectos  
a las organizaciones a construir un modelo para mejorar sus procesos -

### Chain 3

In [21]:
from youtubesearchpython import VideosSearch
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain_core.runnables import RunnableLambda
from langchain.agents import create_agent
import json


llm_model3 = ChatGroq(
     model='llama-3.3-70b-versatile',
    temperature=0,
    max_tokens=300,
    timeout=None,
    max_retries=2,
    verbose=False
)

def parse_videos_search_result(res: str) -> str:
    res_parsed = json.loads(res['messages'][2].content)['result'][0]
    return res_parsed

@dataclass
class UserContext:
    search_terms: str

@tool
def get_video_info(runtime: ToolRuntime[UserContext], search_terms:  str) -> str:
    '''Perform a Youtube search.'''
    search_terms = search_terms
    tool = VideosSearch(search_terms, limit=1)
    res_search = tool.result()
    return res_search

agent3 = create_agent(
    llm_model3,
    tools=[get_video_info],
    context_schema=UserContext,
    system_prompt='You are an assistant that can search videos in Youtube.'
)

chain3 = agent3 | RunnableLambda(parse_videos_search_result)

search_terms = 'Estandarización de procesos, capacitación del personal y auditorías periódicas'
result_search = chain3.invoke(
    {
        'messages': [{
            'role': 'user', 
            'content': 'Extract the link, title and description of the video for the following search terms: {search_terms}.'
        }]
    },
    context=UserContext(search_terms=search_terms)
)

print(f' . Video title: {result_search["title"]}')
print(f' . Channel: {result_search["channel"]["name"]}')
print(f' . Duration: {result_search["duration"]} s')
print(f' . Views: {result_search["viewCount"]["short"]}')
print(f' . URL: {result_search["link"]}')

. Video title: Python for Beginners - Learn Coding with Python in 1 Hour

. Channel: Programming with Mosh

. Duration: 1:00:06 s

. Views: 23M views

. URL: https://www.youtube.com/watch?v=kqtD5dpn9C8

In [22]:
result_search

{'type': 'video',
 'id': 'kqtD5dpn9C8',
 'title': 'Python for Beginners - Learn Coding with Python in 1 Hour',
 'publishedTime': '5 years ago',
 'duration': '1:00:06',
 'viewCount': {'text': '23,795,388 views', 'short': '23M views'},
 'thumbnails': [{'url': 'https://i.ytimg.com/vi/kqtD5dpn9C8/hq720.jpg?sqp=-oaymwEjCOgCEMoBSFryq4qpAxUIARUAAAAAGAElAADIQj0AgKJDeAE=&rs=AOn4CLBZsA0F-g6xPNiHwjpHkONxlfJw7A',
   'width': 360,
   'height': 202},
  {'url': 'https://i.ytimg.com/vi/kqtD5dpn9C8/hq720.jpg?sqp=-oaymwEXCNAFEJQDSFryq4qpAwkIARUAAIhCGAE=&rs=AOn4CLAlOSbnBnoUI-N9v2_o-DYUecBN7A',
   'width': 720,
   'height': 404}],
 'descriptionSnippet': None,
 'channel': {'name': 'Programming with Mosh',
  'id': 'UCWv7vMbMWH4-V0ZXdmDpPBA',
  'thumbnails': [{'url': 'https://yt3.ggpht.com/HCv0fXFEEcD0HRyF0_qR1K7b7qO3KCzmIoyH1DEJYB94CIUFhIE5i2t2IDIPX97W1-DK4hegww=s68-c-k-c0x00ffffff-no-rj',
    'width': 68,
    'height': 68}],
  'link': 'https://www.youtube.com/channel/UCWv7vMbMWH4-V0ZXdmDpPBA'},
 'accessibi

### Roda múltiplas chains

In [23]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

question_about_reviews = 'Can you explain about good practices for the implementation of management systems?'
question_about_topic = 'What review informations do you have about the selected topic?'
selected_topic = 'prácticas concretas para la implementación de sistemas de gestión'
search_terms = 'Estandarización de procesos, capacitación del personal y auditorías periódicas'

multi_stage_chain = RunnableParallel({
        'chain1': RunnableLambda(lambda x: x['input_chain1']) | chain1,
        'chain2': RunnableLambda(lambda x: x['input_chain2']) | chain2,
        'chain3': RunnableLambda(lambda x: x['input_chain3']) | chain3,
    })

res_multi_stage = multi_stage_chain.invoke({
    'input_chain1': {'question_about_reviews': question_about_reviews},
    'input_chain2': {
        'question_about_topic': question_about_topic,
        'selected_topic': selected_topic,
    },
    'input_chain3':
        {
            'messages': [{
                'role': 'user', 
                'content': 'Extract the link, title and description of the video for the following search terms: {search_terms}.',
                'context': search_terms
            }],            
        },
})

print(res_multi_stage)

{
    'chain1': "<think>\nOkay, the user is asking about good practices for implementing management systems. Let me 
start by recalling what management systems typically involve. They're structured frameworks that help organizations
manage processes, resources, and people to achieve objectives. So, the answer should cover key steps and best 
practices.\n\nFirst, leadership commitment is crucial. Without top management support, the system might not get the
necessary resources or attention. Then, aligning the system with the organization's goals ensures it's relevant and
adds value. \n\n",
    'chain2': "<think>\nOkay, the user is asking about review information on specific practices for implementing 
management systems. Let me start by recalling what I know about management system implementation. There are several
standards like ISO 9001 for quality, ISO 14001 for environment, and ISO 45001 for health and safety. Each has their
own guidelines.\n\nFirst, I should mention the key steps involved. Usually, it starts with leadership commitment 
because without top management support, the system won't get the necessary resources. Then there's the gap analysis
to assess the current state versus the standard requirements. Training is important to ensure everyone understands 
their roles.\n\nNext, documentation is crucial. Creating policies, procedures, and work instructions that align 
with the standard. Implementation involves integrating these into daily operations. Internal audits help check 
compliance and identify areas for improvement. Management reviews are part of this to ensure the system remains 
effective.\n\nContinuous improvement through PDCA (Plan-Do-Check-Act) cycles is a common practice. Also, employee 
involvement is key because they need to be engaged and understand the system's benefits. Risk-based thinking is 
another aspect, especially in newer standards like ISO 9001:2015.\n\nI should also consider mentioning the 
importance of selecting the right certification body if they're going for certification. Maybe touch on common 
challenges like resistance to change or lack of resources. References to specific standards and frameworks would 
add credibility.\n\nWait, the user wants concise information. Let me structure this into clear points without 
getting too detailed. Make sure to cover the main practices and maybe reference the ISO standards. Avoid jargon 
where possible, but since it's about management systems, some terms are necessary. Check if there's any recent 
trends or updates in the field, but since the question is about established practices, maybe stick to the 
fundamentals.\n</think>\n\n- **Answer**: Review information on specific practices for implementing management 
systems includes:  \n  1. **Leadership Commitment**: Top management must actively support and drive the system.  \n
2. **Gap Analysis**: Assess current processes against standard requirements (e.g., ISO 9001, ISO 14001).  \n  3. 
**Documentation**: Develop clear policies, procedures, and work instructions aligned with the standard.  \n  4. 
**Training**: Ensure employee understanding of roles and system requirements.  \n  5. **Integration**: Embed the 
system into daily operations and decision",
    'chain3': {
        'type': 'video',
        'id': 'kqtD5dpn9C8',
        'title': 'Python for Beginners - Learn Coding with Python in 1 Hour',
        'publishedTime': '5 years ago',
        'duration': '1:00:06',
        'viewCount': {'text': '23,795,388 views', 'short': '23M views'},
        'thumbnails': [
            {
                'url': 
'https://i.ytimg.com/vi/kqtD5dpn9C8/hq720.jpg?sqp=-oaymwEjCOgCEMoBSFryq4qpAxUIARUAAAAAGAElAADIQj0AgKJDeAE=&rs=AOn4C
LBZsA0F-g6xPNiHwjpHkONxlfJw7A',
                'width': 360,
                'height': 202
            },
            {
                'url': 
'https://i.ytimg.com/vi/kqtD5dpn9C8/hq720.jpg?sqp=-oaymwEXCNAFEJQDSFryq4qpAwkIARUAAIhCGAE=&rs=AOn4CLAlOSbnBnoUI-N9v
2_o-DYUecBN7A',
           